**CitizenTech AI Assitent**
---

*Integrantes*<br>

---

Maria Eduarda Rocha Benjamin RM: <br>
Pedro Henrique Neves RM: 571382


---

In [67]:
%pip install -q --upgrade \
  langchain \
  langchain-core \
  langchain-community \
  langchain-classic \
  langchain-chroma \
  langchain-ollama \
  langchain-huggingface \
  chromadb \
  "transformers>=4.45.0,<5.17.0" \
  "sentence-transformers>=3,<6" \
  huggingface-hub \
  tokenizers \
  gradio \
  gradio-client \
  gTTS \
  python-dotenv


[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


---
## Integração com outras integras


In [68]:
import csv, io, time
import os
import sys
from contextlib import redirect_stdout

# --- 1. CONFIGURAÇÃO DE CAMINHOS PARA O SRC LAYOUT ---
# Como o notebook está dentro de 'src', subimos um nível para achar a raiz
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == 'src':
    PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# --- 2. IMPORTS (Agora usando o pacote 'src') ---
from src.base_conhecimento import BaseConhecimento
from src.orientacao import Orientacao
from src.modulo_decisao import ModuloDecisao
from src.utils import normalizar
from src.auth import _hash, _credenciais_esperadas
from src.fila_virtual import exibir_status_fila
from src.modelagem_urbana import exibir_distribuicao, consultar_regiao

# --- 3. CARREGAR BASE (Voltando um nível para achar a pasta data) ---
CAMINHO_BASE = os.path.join(PROJECT_ROOT, "data", "base_conhecimento.csv")

base = BaseConhecimento(CAMINHO_BASE)
base.carregar()
print(f"✅ Base carregada com sucesso: {len(base.orientacoes)} orientações")

# --- 4. HELPERS DO CSV ---
COLS_CSV = ["id_orientacao", "categoria", "titulo", "palavras_chave", "resposta",
            "documentos", "onde", "fonte", "verificado_em", "expressao_booleana"]

def salvar_base_csv():
    os.makedirs(os.path.dirname(CAMINHO_BASE), exist_ok=True)
    with open(CAMINHO_BASE, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLS_CSV, delimiter=";")
        w.writeheader()
        for o in base.orientacoes:
            w.writerow({k: getattr(o, k, "") for k in COLS_CSV})
    base.carregar()

def consultar_orientacao(id_int):
    return next((o for o in base.orientacoes if o.id_orientacao == id_int), None)

def _capturar_saida(func, *args, **kw):
    buf = io.StringIO()
    with redirect_stdout(buf):
        func(*args, **kw)
    return buf.getvalue().strip()

✅ Base carregada com sucesso: 26 orientações


## Histórico, Notas e Indicadores


In [69]:
import csv, time, os

# Caminhos para salvar os históricos
HIST_PATH  = os.path.join(PROJECT_ROOT, "data", "historico.csv")
NOTAS_PATH = os.path.join(PROJECT_ROOT, "data", "notas.csv")
COLS_HIST  = ["data", "origem", "pergunta", "id_orientacao", "encontrou", "ajudou", "tentativa", "tempo"]

def registrar(pergunta, origem, id_orientacao, encontrou, ajudou, tentativa, tempo):
    os.makedirs(os.path.dirname(HIST_PATH), exist_ok=True)
    if os.path.exists(HIST_PATH):
        with open(HIST_PATH, encoding="utf-8") as f:
            if f.readline().strip().split(";") != COLS_HIST:
                os.remove(HIST_PATH)
    novo = not os.path.exists(HIST_PATH)
    with open(HIST_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLS_HIST, delimiter=";")
        if novo: w.writeheader()
        w.writerow({
            "data": time.strftime("%Y-%m-%d %H:%M:%S"),
            "origem": origem,
            "pergunta": pergunta,
            "id_orientacao": id_orientacao,
            "encontrou": encontrou,
            "ajudou": ajudou,
            "tentativa": tentativa,
            "tempo": tempo,
        })

def salvar_nota(nota):
    os.makedirs(os.path.dirname(NOTAS_PATH), exist_ok=True)
    novo = not os.path.exists(NOTAS_PATH)
    with open(NOTAS_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=";")
        if novo: w.writerow(["data", "nota"])
        w.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), nota])

def indicadores(perguntas_sem_solucao):
    try:
        with open(HIST_PATH, encoding="utf-8") as f:
            registros = list(csv.DictReader(f, delimiter=";"))
    except FileNotFoundError:
        return "Nenhum histórico encontrado."
    if not registros: return "Nenhum atendimento registrado."

    notas = []
    if os.path.exists(NOTAS_PATH):
        with open(NOTAS_PATH, encoding="utf-8") as f:
            for l in csv.DictReader(f, delimiter=";"):
                v = (l.get("nota") or "").strip()
                if v.isdigit(): notas.append(int(v))

    total = len(registros)
    ajudou = sum(1 for r in registros if r["ajudou"] == "sim")
    encontrou = sum(1 for r in registros if r["encontrou"] == "sim")
    tempo_total = sum(float(r["tempo"]) for r in registros)
    tentativas = sum(int(r["tentativa"]) for r in registros)
    sem_solucao_csv = [r["pergunta"] for r in registros if r["encontrou"] != "sim"]

    linhas = [
        f"📊 RELATÓRIO DE INDICADORES — {time.strftime('%Y-%m-%d %H:%M:%S')}",
        "─" * 55,
        f"Total de atendimentos registrados .......... {total}",
        f"% respostas que ajudaram .................... {ajudou/total*100:5.1f}%",
        f"% respostas encontradas ..................... {encontrou/total*100:5.1f}%",
        f"Tempo médio por atendimento ................. {tempo_total/total:5.1f}s",
        f"Tentativas médias por atendimento ........... {tentativas/total:5.2f}",
        f"Facilidade de uso (média notas 1-5) ......... {(sum(notas)/len(notas) if notas else 0):.2f}",
        "", f"Perguntas sem solução nesta sessão: {len(perguntas_sem_solucao)}",
    ]
    for p in perguntas_sem_solucao: linhas.append(f"  • {p[:80]}")
    if sem_solucao_csv:
        linhas.append("\nSem solução no histórico (últimas 5):")
        for p in sem_solucao_csv[-5:]: linhas.append(f"  • {p[:80]}")
    return "\n".join(linhas)

## 1. Base documental e chunking e Embeddings e ChromaDB

In [70]:
import csv, io, time, os, sys
from contextlib import redirect_stdout

# --- 1. CONFIGURAÇÃO DE CAMINHOS ---
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(CURRENT_DIR) if os.path.basename(CURRENT_DIR) == 'src' else CURRENT_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.base_conhecimento import BaseConhecimento
from src.orientacao import Orientacao
from src.modulo_decisao import ModuloDecisao
from src.utils import normalizar
from src.auth import _hash, _credenciais_esperadas
from src.fila_virtual import exibir_status_fila
from src.modelagem_urbana import exibir_distribuicao, consultar_regiao

CAMINHO_BASE = os.path.join(PROJECT_ROOT, "data", "base_conhecimento.csv")
base = BaseConhecimento(CAMINHO_BASE)
base.carregar()
print(f"✅ Base carregada: {len(base.orientacoes)} orientações")

COLS_CSV = ["id_orientacao", "categoria", "titulo", "palavras_chave", "resposta", "documentos", "onde", "fonte", "verificado_em", "expressao_booleana"]

def salvar_base_csv():
    os.makedirs(os.path.dirname(CAMINHO_BASE), exist_ok=True)
    with open(CAMINHO_BASE, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLS_CSV, delimiter=";")
        w.writeheader()
        for o in base.orientacoes:
            w.writerow({k: getattr(o, k, "") for k in COLS_CSV})
    base.carregar()

def consultar_orientacao(id_int):
    return next((o for o in base.orientacoes if o.id_orientacao == id_int), None)

def _capturar_saida(func, *args, **kw):
    buf = io.StringIO()
    with redirect_stdout(buf):
        func(*args, **kw)
    return buf.getvalue().strip()

# --- HISTÓRICO ---
HIST_PATH  = os.path.join(PROJECT_ROOT, "data", "historico.csv")
NOTAS_PATH = os.path.join(PROJECT_ROOT, "data", "notas.csv")
COLS_HIST  = ["data", "origem", "pergunta", "id_orientacao", "encontrou", "ajudou", "tentativa", "tempo"]

def registrar(pergunta, origem, id_orientacao, encontrou, ajudou, tentativa, tempo):
    os.makedirs(os.path.dirname(HIST_PATH), exist_ok=True)
    if os.path.exists(HIST_PATH):
        with open(HIST_PATH, encoding="utf-8") as f:
            if f.readline().strip().split(";") != COLS_HIST:
                os.remove(HIST_PATH)
    novo = not os.path.exists(HIST_PATH)
    with open(HIST_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLS_HIST, delimiter=";")
        if novo: w.writeheader()
        w.writerow({
            "data": time.strftime("%Y-%m-%d %H:%M:%S"), "origem": origem, "pergunta": pergunta,
            "id_orientacao": id_orientacao, "encontrou": encontrou, "ajudou": ajudou,
            "tentativa": tentativa, "tempo": tempo,
        })

def salvar_nota(nota):
    os.makedirs(os.path.dirname(NOTAS_PATH), exist_ok=True)
    novo = not os.path.exists(NOTAS_PATH)
    with open(NOTAS_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=";")
        if novo: w.writerow(["data", "nota"])
        w.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), nota])

def indicadores(perguntas_sem_solucao):
    try:
        with open(HIST_PATH, encoding="utf-8") as f:
            registros = list(csv.DictReader(f, delimiter=";"))
    except FileNotFoundError:
        return "Nenhum histórico encontrado."
    if not registros: return "Nenhum atendimento registrado."

    notas = []
    if os.path.exists(NOTAS_PATH):
        with open(NOTAS_PATH, encoding="utf-8") as f:
            for l in csv.DictReader(f, delimiter=";"):
                v = (l.get("nota") or "").strip()
                if v.isdigit(): notas.append(int(v))

    total = len(registros)
    ajudou = sum(1 for r in registros if r["ajudou"] == "sim")
    encontrou = sum(1 for r in registros if r["encontrou"] == "sim")
    tempo_total = sum(float(r["tempo"]) for r in registros)
    tentativas = sum(int(r["tentativa"]) for r in registros)
    sem_solucao_csv = [r["pergunta"] for r in registros if r["encontrou"] != "sim"]

    linhas = [
        f"📊 RELATÓRIO DE INDICADORES — {time.strftime('%Y-%m-%d %H:%M:%S')}", "─" * 55,
        f"Total de atendimentos .......... {total}",
        f"% respostas que ajudaram ....... {ajudou/total*100:5.1f}%",
        f"% respostas encontradas ........ {encontrou/total*100:5.1f}%",
        f"Tempo médio .................... {tempo_total/total:5.1f}s",
        f"Tentativas médias .............. {tentativas/total:5.2f}",
        f"Facilidade de uso (1-5) ........ {(sum(notas)/len(notas) if notas else 0):.2f}",
        "", f"Perguntas sem solução nesta sessão: {len(perguntas_sem_solucao)}",
    ]
    for p in perguntas_sem_solucao: linhas.append(f"  • {p[:80]}")
    return "\n".join(linhas)

✅ Base carregada: 26 orientações


## 2. Pipeline RAG com contexto

In [71]:
import torch

def _get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if dtype is None:
        dtype = self.dtype
    if device is None:
        device = attention_mask.device
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        raise ValueError(f"Dimensão inesperada de attention_mask: {attention_mask.dim()}")
    extended_attention_mask = extended_attention_mask.to(dtype=dtype, device=device)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
    return extended_attention_mask

# Aplique na instância/classe carregada do NomicBertModel (ex.: model.get_extended_attention_mask = ...)

## 3. Agente com ferramentas

In [78]:
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_classic.agents import AgentExecutor, create_react_agent

# --- 1. Carregar Documentos ---
caminhos = [
    {"caminho": os.path.join(PROJECT_ROOT, "data", "doc_real.txt"), "doc_id": "doc_real"},
    {"caminho": os.path.join(PROJECT_ROOT, "data", "doc_armadilha.txt"), "doc_id": "doc_armadilha"}
]
docs = []
for item in caminhos:
    if os.path.exists(item["caminho"]):
        loader = TextLoader(item["caminho"], encoding="utf-8")
        documentos = loader.load()
        for doc in documentos: doc.metadata = {"doc_id": item["doc_id"]}
        docs.extend(documentos)

if not docs:
    print("⚠️ Criando texto de exemplo para o ChromaDB não falhar.")
    docs = [Document(page_content="Texto de exemplo sobre a carteira do idoso.", metadata={"doc_id": "exemplo"})]

# --- 2. Chunking e Embeddings ---
print("Carregando modelo de embeddings...")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_finais = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", 
    model_kwargs={'device': 'cpu'}
)

# --- 3. ChromaDB ---
vectorstore = Chroma.from_documents(
    documents=chunks_finais, 
    embedding=embeddings, 
    collection_name="citizen_assist_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# --- 4. CONEXÃO COM OLLAMA LOCAL ---
print("Conectando ao Ollama Local...")
llm = ChatOllama(
    model="llama3.2", # Garanta que o aplicativo do Ollama está aberto no Windows
    temperature=0.1
)

# --- 5. Definição das Ferramentas ---
@tool
def consultar_base_conhecimento(pergunta: str) -> str:
    """Use esta ferramenta para buscar informações na base de conhecimento sobre a carteira do idoso e biometria."""
    docs = retriever.invoke(pergunta)
    return "\n\n".join([d.page_content for d in docs])

@tool
def calcular_tempo_espera(pessoas_na_fila: int) -> str:
    """Calcula o tempo estimado de espera na fila (5 min por pessoa)."""
    return f"Com {pessoas_na_fila} pessoas, a espera é de {pessoas_na_fila * 5} minutos."

@tool
def verificar_necessidade_biometria(idade: int) -> str:
    """Verifica se um idoso precisa ir presencialmente ao posto (80+ anos)."""
    if idade >= 80: return f"Por ter {idade} anos, você DEVE ir ao posto anualmente."
    return f"Por ter {idade} anos, você está dispensado."

tools = [consultar_base_conhecimento, calcular_tempo_espera, verificar_necessidade_biometria]

# --- 6. Agente ReAct ---
react_template = """Responda às perguntas do usuário da melhor forma possível. Você tem acesso às seguintes ferramentas:
{tools}

Use o seguinte formato:
Question: a pergunta que você deve responder
Thought: você deve sempre pensar sobre o que fazer
Action: a ação a ser tomada, deve ser uma de [{tool_names}]
Action Input: a entrada para a ação
Observation: o resultado da ação
Thought: Eu agora sei a resposta final
Final Answer: a resposta final para a pergunta original

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_template)
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False, handle_parsing_errors=True, max_iterations=5)

print("✅ Agente de IA configurado com sucesso no modo LOCAL!")

Carregando modelo de embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1839.31it/s]


Conectando ao Ollama Local...
✅ Agente de IA configurado com sucesso no modo LOCAL!


## 5. Interface Gradio

In [86]:
import gradio as gr
from gtts import gTTS
import os
import time
import uuid

# Função de áudio
def gerar_audio(texto):
    if not texto or len(texto.strip()) < 3:
        return None
    try:
        texto_limpo = texto.replace("*", "").replace("#", "").replace("-", " ")
        tts = gTTS(text=texto_limpo, lang='pt', tld='com.br')
        filename = f"audio_{uuid.uuid4().hex}.mp3" 
        caminho = os.path.abspath(os.path.join(PROJECT_ROOT, "data", filename)).replace("\\", "/")
        os.makedirs(os.path.dirname(caminho), exist_ok=True)
        tts.save(caminho)
        if os.path.exists(caminho) and os.path.getsize(caminho) > 0:
            return caminho
        return None
    except Exception as e:
        print(f"❌ Erro ao gerar áudio: {e}")
        return None

class ControladorGradioCitizen:
    def __init__(self):
        self.logado = False
        self.orientacao_ativa = None
        self.perguntas_sem_solucao_sessao = []

    def login_equipe(self, usuario, senha):
        u_hash, s_hash = _credenciais_esperadas()
        ok = (_hash((usuario or "").strip()) == u_hash and _hash((senha or "").strip()) == s_hash)
        self.logado = ok
        msg = ("🟢 Acesso autorizado." if ok else "🔴 Credenciais inválidas.")
        return (msg, gr.update(visible=ok), gr.update(visible=not ok), gr.update(visible=ok), gr.update(visible=not ok))

    def logout_equipe(self):
        self.logado = False
        return ("⚪ Sessão encerrada.", gr.update(visible=False), gr.update(visible=False), gr.update(visible=False), gr.update(visible=True))

    def acao_1_perguntar(self, pergunta):
        self.orientacao_ativa = None
        texto = (pergunta or "").strip()
        if not texto: 
            return "Digite uma pergunta.", None, gr.update(visible=False), gr.update(visible=False), "", ""

        if "aplicar_guardrails" in globals():
            r = aplicar_guardrails(texto)
            if not r.get("permitido", True):
                msg_bloqueio = r.get("resposta_segura", "Bloqueado.")
                audio_bloqueio = gerar_audio(msg_bloqueio)
                return (msg_bloqueio, audio_bloqueio, gr.update(visible=False), gr.update(visible=False), "", "")

        t0 = time.time()
        resultados = base.buscar(texto)
        orientacao = resultados[0][1] if resultados else None
        pontos = resultados[0][0] if resultados else 0

        try:
            resultado_ia = agent_executor.invoke({"input": texto})
            resposta_ia = resultado_ia.get("output", "")
        except Exception as e:
            resposta_ia = f"(IA indisponível: {e})"

        tempo = round(time.time() - t0, 1)
        resposta_final = ""

        if resposta_ia and "Não sei" not in resposta_ia:
            resposta_final += f"🤖 **Resposta do Assistente IA:**\n{resposta_ia}\n\n"

        if orientacao and pontos >= 3:
            resposta_final += f"---\n📚 **Detalhes da Base de Conhecimento:**\n{_capturar_saida(orientacao.exibir)}\n"
            tem_regra = bool(getattr(orientacao, "expressao_booleana", "").strip())
            
            texto_falado = f"Você perguntou: {texto}. {resposta_final}"
            audio = gerar_audio(texto_falado)
            
            if tem_regra:
                self.orientacao_ativa = orientacao
                aviso = ("Deseja verificar se você tem direito? Selecione **sim** abaixo.\n⚠️ Este verificador **não coleta dados pessoais**.")
                registrar(texto, "hibrido", orientacao.id_orientacao, "sim", "pendente", 1, tempo)
                return (resposta_final, audio, gr.update(visible=True), gr.update(visible=False), aviso, "")
            
            registrar(texto, "hibrido", orientacao.id_orientacao, "sim", "pendente", 1, tempo)
            return (resposta_final, audio, gr.update(visible=False), gr.update(visible=False), "", "")

        if not resposta_ia and not orientacao:
            self.perguntas_sem_solucao_sessao.append(texto)
            registrar(texto, "hibrido", "N/A", "nao", "nao", 1, tempo)
            resposta_final = "Não encontrei uma resposta exata. Tente navegar pelas categorias."

        texto_falado = f"Você perguntou: {texto}. {resposta_final}"
        audio = gerar_audio(texto_falado)
        return (resposta_final, audio, gr.update(visible=False), gr.update(visible=False), "", "")

    def acao_elegibilidade_escolha(self, escolha):
        if (escolha or "").strip().lower() != "sim" or self.orientacao_ativa is None:
            return gr.update(visible=False), "", None
            
        try:
            m = ModuloDecisao(self.orientacao_ativa)
            expr = m.obter_expressao_booleana()
            vars_ = m.obter_variaveis()
            
            if not vars_: return gr.update(visible=True), "Nenhum critério encontrado.", None
                
            expr_formatada = " ".join(expr) if isinstance(expr, list) else str(expr)
            instrucao = ("Regra: `" + expr_formatada + "`\n\nPreencha uma linha por critério com `sim` ou `nao`:\n" + "\n".join(f"{v}=sim" for v in vars_))
            
            texto_falado = f"A regra deste benefício é: {expr_formatada}. Para verificar, digite no campo os critérios, escrevendo sim ou nao para cada um: {vars_}."
            audio_instrucao = gerar_audio(texto_falado)
            
            return gr.update(visible=True), instrucao, audio_instrucao
        except Exception as e:
            print(f"❌ ERRO ELEGIBILIDADE: {e}")
            return gr.update(visible=True), f"Erro ao carregar critérios: {e}", None

    def acao_elegibilidade_calcular(self, respostas_texto):
        if self.orientacao_ativa is None: return "Nenhuma orientação ativa."
        m = ModuloDecisao(self.orientacao_ativa)
        m.obter_expressao_booleana()
        vars_ = m.obter_variaveis()
        mapa = {}
        for linha in (respostas_texto or "").splitlines():
            if "=" in linha:
                k, v = linha.split("=", 1)
                k = k.strip().lower().replace(" ", "_")
                v = normalizar(v).strip()
                if v in {"sim", "nao"}: mapa[k] = 1 if v == "sim" else 0
        for var in vars_:
            if var not in mapa: return f"Informe `{var}=sim` ou `{var}=nao`."
            m.valores_variavel[var] = mapa[var]
        m.substituir_variavel_por_valor()
        res = m.resolver_expressao()
        veredito = ("RESULTADO: Você PODE solicitar este benefício." if res == 1 else "RESULTADO: Você NÃO é elegível." if res == 0 else "Não foi possível calcular.")
        registrar(f"Elegibilidade: {self.orientacao_ativa.titulo}", "modulo_decisao", self.orientacao_ativa.id_orientacao, "sim", "sim" if res == 1 else "nao", 1, 0)
        return veredito

    def carregar_categorias(self):
        try:
            cats = base.categorias()
            return gr.update(choices=cats, value=cats[0] if cats else None)
        except Exception as e:
            print(f"❌ ERRO AO CARREGAR CATEGORIAS: {e}")
            return gr.update(choices=[], value=None)

    def acao_2_categoria(self, categoria):
        if not categoria: 
            return "Selecione uma categoria.", None
        try:
            itens = base.por_categoria(categoria)
            if itens:
                texto = "\n\n".join(_capturar_saida(i.exibir) for i in itens)
                return texto, gerar_audio(f"Você selecionou a categoria {categoria}. {texto}")
            else:
                return "Nenhuma orientação encontrada.", gerar_audio("Nenhuma orientação encontrada.")
        except Exception as e:
            print(f"❌ ERRO NA CATEGORIA: {e}")
            return f"Erro: {e}", None

    def carregar_palavras(self):
        try:
            palavras = set()
            for o in base.orientacoes:
                for p in str(o.palavras_chave).split("|"):
                    if p.strip(): palavras.add(p.strip())
            ordenadas = sorted(palavras)
            return gr.update(choices=ordenadas, value=ordenadas[0] if ordenadas else None)
        except Exception as e:
            print(f"❌ ERRO AO CARREGAR PALAVRAS: {e}")
            return gr.update(choices=[], value=None)

    def acao_3_palavra(self, palavra):
        if not palavra: 
            return "Escolha uma palavra-chave.", None
        try:
            achados = [o for o in base.orientacoes if palavra.lower() in str(o.palavras_chave).lower()]
            if achados:
                texto = "\n\n".join(_capturar_saida(o.exibir) for o in achados)
                return texto, gerar_audio(f"Você pesquisou por {palavra}. {texto}")
            else:
                return f"Nenhuma orientação encontrada para '{palavra}'.", gerar_audio("Nenhuma orientação encontrada.")
        except Exception as e:
            print(f"❌ ERRO NA PALAVRA-CHAVE: {e}")
            return f"Erro: {e}", None

    def acao_4_ordenar(self, criterio, inverso):
        chaves = {"1) Número": lambda o: o.id_orientacao, "2) Título": lambda o: normalizar(o.titulo), "3) Assunto": lambda o: normalizar(o.categoria)}
        if criterio not in chaves: return "Opção inválida."
        ordenadas = sorted(base.orientacoes, key=chaves[criterio], reverse=bool(inverso))
        return "\n\n".join(f"#{o.id_orientacao} — {o.titulo} [{o.categoria}]" for o in ordenadas)

    def acao_5_listar(self):
        return "\n\n".join(_capturar_saida(o.exibir) for o in base.orientacoes)

    def acao_6_fila(self):
        texto = _capturar_saida(exibir_status_fila)
        return texto, gerar_audio(f"Resultado da consulta de fila virtual: {texto}")

    def acao_7_energia(self, kwh):
        try:
            from src import energia
            v = float(kwh)
            texto = f"Classificação: {energia.classificar_consumo(v)}. Conta estimada: R$ {energia.calcular_conta(v):.2f}"
            return texto, gerar_audio(f"Sobre o consumo de {kwh} kWh: {texto}")
        except Exception as e: 
            return f"Indisponível ({e}).", None

    def acao_8_onibus(self, modo, regiao):
        if modo == "1) Ver todas as regiões": 
            texto = _capturar_saida(exibir_distribuicao)
            return texto, gerar_audio(f"Distribuição de ônibus em todas as regiões: {texto}")
        else:
            texto = _capturar_saida(consultar_regiao, regiao or "Norte")
            return texto, gerar_audio(f"Ônibus na região {regiao or 'Norte'}: {texto}")

    def acao_9_cadastrar(self, id_str, categoria, titulo, palavras, resposta, documentos, onde, fonte, verificado_em, expressao):
        if not self.logado: return "Acesso negado."
        try: id_int = int(str(id_str).strip())
        except: return "ID inválido."
        if consultar_orientacao(id_int) is not None: return f"ID {id_int} já existe."
        base.orientacoes.append(Orientacao(id_orientacao=id_int, categoria=categoria.strip(), titulo=titulo.strip(), palavras_chave=(palavras or "").strip(), resposta=(resposta or "").strip(), documentos=(documentos or "").strip(), onde=(onde or "").strip(), fonte=(fonte or "").strip(), verificado_em=(verificado_em or "").strip(), expressao_booleana=(expressao or "").strip()))
        salvar_base_csv()
        return f"✅ Orientação #{id_int} cadastrada."

    def acao_10_consultar_id(self, id_str):
        if not self.logado: return "Acesso negado."
        try: o = consultar_orientacao(int(str(id_str).strip()))
        except: return "ID inválido."
        return _capturar_saida(o.exibir) if o else f"Nada encontrado."

    def acao_11_atualizar(self, id_str, titulo, resposta, expressao):
        if not self.logado: return "Acesso negado."
        try: id_int = int(str(id_str).strip())
        except: return "ID inválido."
        o = consultar_orientacao(id_int)
        if o is None: return f"#{id_int} não encontrada."
        if (titulo or "").strip(): o.titulo = titulo.strip()
        if (resposta or "").strip(): o.resposta = resposta.strip()
        if (expressao or "").strip(): o.expressao_booleana = expressao.strip()
        salvar_base_csv()
        return f"✅ #{id_int} atualizada."

    def acao_12_excluir(self, id_str, confirmar):
        if not self.logado: return "Acesso negado."
        if not confirmar: return "Marque 'Confirmar exclusão'."
        try: id_int = int(str(id_str).strip())
        except: return "ID inválido."
        antes = len(base.orientacoes)
        base.orientacoes = [o for o in base.orientacoes if o.id_orientacao != id_int]
        if len(base.orientacoes) == antes: return f"#{id_int} não encontrada."
        salvar_base_csv()
        return f"✅ #{id_int} excluída."

    def acao_salvar_nota(self, nota):
        try:
            n = int(nota)
            if not 1 <= n <= 5: return "Nota fora da faixa (1 a 5)."
        except: return "Valor inválido."
        salvar_nota(n)
        return f"✅ Nota {n} registrada."

    def acao_indicadores(self):
        return indicadores(self.perguntas_sem_solucao_sessao)

In [87]:
ctrl = ControladorGradioCitizen()

texto_instrucoes = (
    "Bem-vindo ao Citizen Assist AI. "
    "Esta é uma interface acessível. "
    "Use as abas no topo da tela para navegar entre as funções. "
    "Na aba de perguntas, digite sua dúvida e clique em buscar. O assistente lerá a resposta em voz alta para você."
)

print("Gerando áudio de boas-vindas...")
caminho_audio_inicial = gerar_audio(texto_instrucoes)

def gerar_audio_inicial():
    return gerar_audio(texto_instrucoes)

with gr.Blocks(title="Citizen Assist AI Acessível", theme=gr.themes.Soft()) as app:
    
    # Topo da página
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("# 🤖 Bem Vindo ao Citizen Assist AI")
            gr.Markdown("Sistema Híbrido de Atendimento (Base de Dados + IA Generativa)")
            
            gr.Markdown(
                "> ♿ **Instruções de Acessibilidade:** "
                "Se você utiliza um leitor de tela, o foco do teclado já está no botão abaixo. "
                "Pressione **ENTER** para ouvir as instruções em voz alta. "
                "Esse primeiro clique liberará o áudio para todas as outras funções do sistema."
            )
            
            audio_pagina = gr.Audio(
                value=caminho_audio_inicial, 
                label="🔊 Áudio das Instruções Iniciais", 
                visible=True, 
                autoplay=True, 
                format="mp3"
            )
            
            btn_ler_pagina = gr.Button(
                "🔊 CLIQUE AQUI PARA OUVIR AS INSTRUÇÕES INICIAIS", 
                variant="primary", 
                size="lg"
            )
            
        with gr.Column(scale=1):
            btn_abrir_login = gr.Button("🔒 Acesso Restrito (Equipe)", variant="secondary")
            
            with gr.Group(visible=False) as grp_login:
                with gr.Row():
                    inp_user = gr.Textbox(label="Usuário", placeholder="equipe", scale=1)
                    inp_pass = gr.Textbox(label="Senha", type="password", scale=1)
                with gr.Row():
                    btn_login  = gr.Button("Entrar", variant="primary", scale=1)
                    btn_cancelar = gr.Button("Cancelar", scale=1)
            
            btn_logout = gr.Button("Sair do Sistema", visible=False, variant="stop")
            lbl_auth = gr.Markdown("")

    # Abas principais
    with gr.Tabs():
        with gr.Tab("1. Perguntar"):
            txt_perg = gr.Textbox(
                label="Digite sua dúvida (ou dite usando o microfone do seu sistema)", 
                placeholder="Ex: Quais os documentos para a carteira do idoso?"
            )
            btn_perg = gr.Button("Buscar Resposta (IA + Base)", variant="primary")
            out_perg = gr.Markdown("A resposta da IA e da Base aparecerão aqui.")
            
            audio_resposta = gr.Audio(label="🔊 Ouça a resposta", visible=True, autoplay=True, format="mp3")
            
            with gr.Group(visible=False) as grp_elig_escolha:
                lbl_elig_aviso = gr.Markdown("")
                rad_elig = gr.Radio(choices=["nao", "sim"], value="nao", label="Deseja verificar a elegibilidade para este benefício?")
            
            with gr.Group(visible=False) as grp_elig_form:
                lbl_elig_vars = gr.Markdown("")
                audio_elig_instrucao = gr.Audio(label="🔊 Instruções da Regra", visible=True, autoplay=True, format="mp3")
                txt_elig_in = gr.Textbox(label="Critérios (responda criterio=sim ou criterio=nao)", lines=3)
                btn_elig = gr.Button("Verificar Direito")
                out_elig = gr.Textbox(label="Resultado da Elegibilidade", lines=4)
                
            btn_perg.click(fn=ctrl.acao_1_perguntar, inputs=[txt_perg], outputs=[out_perg, audio_resposta, grp_elig_escolha, grp_elig_form, lbl_elig_aviso, out_elig])
            rad_elig.change(fn=ctrl.acao_elegibilidade_escolha, inputs=[rad_elig], outputs=[grp_elig_form, lbl_elig_vars, audio_elig_instrucao])
            btn_elig.click(fn=ctrl.acao_elegibilidade_calcular, inputs=[txt_elig_in], outputs=[out_elig])

        with gr.Tab("2. Categorias"):
            with gr.Row():
                dd_cat = gr.Dropdown(label="Selecione a Categoria", choices=[], scale=3)
                btn_load_cat = gr.Button("Atualizar Lista", scale=1)
                btn_run_cat  = gr.Button("Exibir Orientações", variant="primary", scale=1)
            out_cat = gr.Markdown("")
            audio_cat = gr.Audio(label="🔊 Áudio da Categoria", visible=True, autoplay=True, format="mp3")
            btn_load_cat.click(fn=ctrl.carregar_categorias, outputs=[dd_cat])
            btn_run_cat.click(fn=ctrl.acao_2_categoria, inputs=[dd_cat], outputs=[out_cat, audio_cat])

        with gr.Tab("3. Palavras-chave"):
            with gr.Row():
                dd_pal = gr.Dropdown(label="Selecione a Palavra-chave", choices=[], scale=3)
                btn_load_pal = gr.Button("Atualizar Lista", scale=1)
                btn_run_pal  = gr.Button("Pesquisar", variant="primary", scale=1)
            out_pal = gr.Markdown("")
            audio_pal = gr.Audio(label="🔊 Áudio da Pesquisa", visible=True, autoplay=True, format="mp3")
            btn_load_pal.click(fn=ctrl.carregar_palavras, outputs=[dd_pal])
            btn_run_pal.click(fn=ctrl.acao_3_palavra, inputs=[dd_pal], outputs=[out_pal, audio_pal])

        with gr.Tab("4. Ordenar"):
            with gr.Row():
                dd_ord = gr.Dropdown(choices=["1) Número", "2) Título", "3) Assunto"], value="1) Número", label="Critério de Ordenação", scale=2)
                chk_inv = gr.Checkbox(label="Ordem Inversa?", scale=1)
                btn_ord = gr.Button("Ordenar", variant="primary", scale=1)
            out_ord = gr.Markdown("")
            btn_ord.click(fn=ctrl.acao_4_ordenar, inputs=[dd_ord, chk_inv], outputs=[out_ord])

        with gr.Tab("5. Listar"):
            btn_lis = gr.Button("Listar Todas as Orientações", variant="primary")
            out_lis = gr.Markdown("")
            btn_lis.click(fn=ctrl.acao_5_listar, outputs=[out_lis])

        with gr.Tab("6. Fila"):
            btn_fila = gr.Button("Consultar Status da Fila Virtual", variant="primary")
            out_fila = gr.Textbox(label="Status da Fila", lines=10)
            audio_fila = gr.Audio(label="🔊 Áudio da Fila", visible=False, autoplay=True, format="mp3")
            btn_fila.click(fn=ctrl.acao_6_fila, outputs=[out_fila, audio_fila])

        with gr.Tab("7. Energia"):
            with gr.Row():
                num_kwh = gr.Number(label="Consumo de Energia (kWh/mês)", value=200, scale=2)
                btn_en = gr.Button("Consultar Consumo", variant="primary", scale=1)
            out_en = gr.Markdown("")
            audio_en = gr.Audio(label="🔊 Áudio da Energia", visible=False, autoplay=True, format="mp3")
            btn_en.click(fn=ctrl.acao_7_energia, inputs=[num_kwh], outputs=[out_en, audio_en])

        with gr.Tab("8. Ônibus"):
            with gr.Row():
                rad_on = gr.Radio(choices=["1) Ver todas as regiões", "2) Região específica"], value="1) Ver todas as regiões", label="Modo de Consulta", scale=2)
                dd_reg = gr.Dropdown(choices=["Norte","Sul","Leste","Oeste"], value="Norte", label="Região", scale=1)
                btn_on = gr.Button("Consultar", variant="primary", scale=1)
            out_on = gr.Markdown("")
            audio_on = gr.Audio(label="🔊 Áudio dos Ônibus", visible=False, autoplay=True, format="mp3")
            btn_on.click(fn=ctrl.acao_8_onibus, inputs=[rad_on, dd_reg], outputs=[out_on, audio_on])

        with gr.Tab("9-12. Equipe", visible=False) as tab_equipe:
            out_adm = gr.Textbox(label="Status da Operação", lines=2, interactive=False)
            
            with gr.Accordion("9. Cadastrar Nova Orientação", open=False):
                with gr.Row():
                    c_id = gr.Textbox(label="ID (Número)", scale=1); c_cat = gr.Textbox(label="Categoria", scale=2); c_tit = gr.Textbox(label="Título", scale=3)
                with gr.Row():
                    c_pal = gr.Textbox(label="Palavras-chave (separadas por |)", scale=2); c_doc = gr.Textbox(label="Documentos", scale=1)
                with gr.Row():
                    c_onde = gr.Textbox(label="Onde", scale=1); c_fonte = gr.Textbox(label="Fonte", scale=2); c_data = gr.Textbox(label="Verificado em", scale=1)
                c_expr = gr.Textbox(label="Expressão booleana (opcional)")
                c_resp = gr.Textbox(label="Resposta da Orientação", lines=3)
                btn_cad = gr.Button("Cadastrar", variant="primary")
                btn_cad.click(fn=ctrl.acao_9_cadastrar, inputs=[c_id,c_cat,c_tit,c_pal,c_resp,c_doc,c_onde,c_fonte,c_data,c_expr], outputs=[out_adm])

            with gr.Accordion("10. Consultar por ID", open=False):
                with gr.Row():
                    q_id = gr.Textbox(label="ID", scale=1); btn_con = gr.Button("Consultar", variant="primary", scale=2)
                btn_con.click(fn=ctrl.acao_10_consultar_id, inputs=[q_id], outputs=[out_adm])

            with gr.Accordion("11. Atualizar Orientação", open=False):
                with gr.Row():
                    u_id = gr.Textbox(label="ID para atualizar", scale=1); u_tit = gr.Textbox(label="Novo título (opcional)", scale=2); u_expr = gr.Textbox(label="Nova expressão (opcional)", scale=2)
                u_resp = gr.Textbox(label="Nova resposta (opcional)", lines=2)
                btn_atu = gr.Button("Atualizar", variant="primary")
                btn_atu.click(fn=ctrl.acao_11_atualizar, inputs=[u_id,u_tit,u_resp,u_expr], outputs=[out_adm])

            with gr.Accordion("12. Excluir Orientação", open=False):
                with gr.Row():
                    d_id = gr.Textbox(label="ID para excluir", scale=1); d_conf = gr.Checkbox(label="Confirmar exclusão", scale=1); btn_exc = gr.Button("Excluir", variant="stop", scale=1)
                btn_exc.click(fn=ctrl.acao_12_excluir, inputs=[d_id,d_conf], outputs=[out_adm])

        with gr.Tab("13-14. Avaliação"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 13. Nota para o Sistema")
                    num_nota = gr.Number(label="Nota (1 a 5)", value=5, minimum=1, maximum=5, precision=0)
                    btn_nota = gr.Button("Enviar Avaliação")
                    out_nota = gr.Textbox(label="Status", lines=2)
                    btn_nota.click(fn=ctrl.acao_salvar_nota, inputs=[num_nota], outputs=[out_nota])
                with gr.Column():
                    gr.Markdown("### 14. Relatório de Indicadores")
                    btn_ind = gr.Button("Gerar Relatório", variant="primary")
                    out_ind = gr.Textbox(label="Métricas do Sistema", lines=12)
                    btn_ind.click(fn=ctrl.acao_indicadores, outputs=[out_ind])

    # Eventos de Login/Logout
    btn_abrir_login.click(fn=lambda: gr.update(visible=True), outputs=[grp_login])
    btn_cancelar.click(fn=lambda: (gr.update(visible=False), ""), outputs=[grp_login, lbl_auth])
    btn_login.click(fn=ctrl.login_equipe, inputs=[inp_user, inp_pass], outputs=[lbl_auth, tab_equipe, grp_login, btn_logout, btn_abrir_login])
    btn_logout.click(fn=ctrl.logout_equipe, outputs=[lbl_auth, tab_equipe, grp_login, btn_logout, btn_abrir_login])
    
    # Eventos de áudio e carregamento de dados
    btn_ler_pagina.click(fn=gerar_audio_inicial, inputs=[], outputs=[audio_pagina])
    app.load(fn=ctrl.carregar_categorias, inputs=[], outputs=[dd_cat])
    app.load(fn=ctrl.carregar_palavras, inputs=[], outputs=[dd_pal])

    # --- SCRIPT DE ACESSIBILIDADE 1: Joga o foco do teclado direto no botão de áudio inicial ---
    app.load(
        fn=None,
        inputs=None,
        outputs=None,
        js="""
        () => {
            setTimeout(() => {
                const botoes = Array.from(document.querySelectorAll('button'));
                const btnAudio = botoes.find(b => b.innerText.includes('OUVIR AS INSTRUÇÕES'));
                if (btnAudio) { btnAudio.focus(); }
            }, 1500);
        }
        """
    )

    # --- SCRIPT DE ACESSIBILIDADE 2: Joga o foco do teclado no campo "Usuário" quando o login abrir ---
    btn_abrir_login.click(
        fn=None,
        inputs=None,
        outputs=None,
        js="""
        () => {
            setTimeout(() => {
                const inputs = document.querySelectorAll('input');
                const inputUser = Array.from(inputs).find(i => i.placeholder === 'equipe');
                if (inputUser) { inputUser.focus(); }
            }, 500);
        }
        """
    )

print("✅ Interface acessível montada! Abrindo o navegador...")
app.launch(share=False, inbrowser=True, allowed_paths=[os.path.join(PROJECT_ROOT, "data")])

Gerando áudio de boas-vindas...


C:\Users\pedri\AppData\Local\Temp\ipykernel_45212\663338537.py:16: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Citizen Assist AI Acessível", theme=gr.themes.Soft()) as app:


✅ Interface acessível montada! Abrindo o navegador...
* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


---

## 6. Avaliação RAGAS

In [19]:
import csv, os
from datetime import datetime

# Correção do NameError
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(CURRENT_DIR) if os.path.basename(CURRENT_DIR) == 'src' else CURRENT_DIR

RAGAS_CSV_PATH = os.path.join(PROJECT_ROOT, "data", "historico_ragas.csv")
COLS_RAGAS = ["data_avaliacao", "id_pergunta", "tipo", "pergunta", "contexto_recuperado", "resposta_sistema", "ground_truth"]

dataset_avaliacao = [
    {"id": "Q1", "tipo": "simples", "pergunta": "Quais os documentos para tirar a carteira do idoso?", "ground_truth": "RG, CPF e comprovante de residência."},
    {"id": "Q2", "tipo": "simples", "pergunta": "Onde fica o posto de atendimento?", "ground_truth": "Na rua central, número 100."},
    {"id": "Q3", "tipo": "simples", "pergunta": "Qual o prazo para ficar pronto?", "ground_truth": "Cerca de 15 dias úteis."},
    {"id": "Q4", "tipo": "simples", "pergunta": "Idosos acima de 80 anos precisam ir ao posto?", "ground_truth": "Sim, uma vez por ano para biometria."},
    {"id": "Q5", "tipo": "armadilha", "pergunta": "Pode fazer tudo pelo celular?", "ground_truth": "Depende da idade."},
    {"id": "Q6", "tipo": "armadilha", "pergunta": "A regra de 2023 ainda vale?", "ground_truth": "Não."},
    {"id": "Q7", "tipo": "armadilha", "pergunta": "Quem tem 79 anos precisa ir presencial?", "ground_truth": "Não."},
    {"id": "Q8", "tipo": "armadilha", "pergunta": "E se eu não puder andar?", "ground_truth": "Atendimento domiciliar."},
]

os.makedirs(os.path.dirname(RAGAS_CSV_PATH), exist_ok=True)
arquivo_existe = os.path.exists(RAGAS_CSV_PATH)

with open(RAGAS_CSV_PATH, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLS_RAGAS, delimiter=";")
    if not arquivo_existe: writer.writeheader()

    for item in dataset_avaliacao:
        resultados = base.buscar(item["pergunta"])
        if resultados:
            pontos, orientacao = resultados[0]
            contexto = orientacao.resposta.replace("\n", " ")
            resposta_sistema = f"[Confiança: {pontos:.1f}] {orientacao.titulo}"
        else:
            contexto = "Nenhum contexto."
            resposta_sistema = "Não encontrei."
            
        writer.writerow({
            "data_avaliacao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "id_pergunta": item["id"], "tipo": item["tipo"], "pergunta": item["pergunta"],
            "contexto_recuperado": contexto, "resposta_sistema": resposta_sistema, "ground_truth": item["ground_truth"]
        })

print(f"🎉 Avaliação concluída! Resultados salvos em: {RAGAS_CSV_PATH}")

🎉 Avaliação concluída! Resultados salvos em: c:\Users\pedri\PycharmProjects\citizentech_assistent_ai\data\historico_ragas.csv


### 7. Conclusão

O projeto **CitizenTech** estruturou um fluxo de consulta cidadã apoiado em base documental autoral, chunking comparativo, indexação vetorial local no ChromaDB com embeddings `all-MiniLM-L6-v2`, interface interativa em Gradio e guardrails de escopo, LGPD e prompt injection.
Na avaliação da Seção 7, foram definidas 8 perguntas (4 diretas e 4 de armadilha) com `ground_truth` elaborado pelo grupo, além da recuperação de contextos e exportação do arquivo `3_avaliacao_ragas.json`.
Durante a execução no Colab, o pipeline local operou de forma reprodutível, mas a geração de respostas/escores remotos via Ollama Cloud encontrou bloqueio `HTTP 402` ("Add usage credits or subscribe") e a avaliação numérica RAGAS não foi executada pela ausência do pacote `ragas` no ambiente.
Nenhuma métrica ou saída de modelo foi estimada ou inventada, mantendo a transparência técnica sobre os limites do ambiente de teste.
Como próximos passos, a integração com um endpoint LLM ativo e a instalação do `ragas` permitirão consolidar os escores quantitativos sobre o conjunto de 8 perguntas.
O projeto demonstra como combinar recuperação semântica, validação de entrada e governança de dados para apoiar orientações ao cidadão com rastreabilidade de fontes.